# IMAS Reader Basics

This notebook is for a downstream user who only wants to read SOLEDGE-HDG IMAS `.nc` files.

It assumes only:

- `imas-python`
- `numpy`
- `matplotlib`

It follows the three export patterns used in the IMAS export tutorial:

- one steady-state case per file
- one bundled scan with one IDS occurrence per simulation
- one full discharge with multiple time indices in one file


In [ ]:
import json

import imas
import matplotlib.pyplot as plt
import numpy as np


## Small helpers

These helper functions keep the examples below compact.


In [ ]:
def load_ids(db_path, occurrence=0):
    with imas.DBEntry(db_path, "r") as entry:
        summary = entry.get("summary", occurrence)
        equilibrium = entry.get("equilibrium", occurrence)
        plasma = entry.get("plasma_profiles", occurrence)
    return summary, equilibrium, plasma


def equilibrium_slice(equilibrium, time_index=0):
    params = json.loads(str(equilibrium.code.parameters))
    nr, nz = params["grid_shape"]
    eq_ggd = equilibrium.time_slice[time_index].ggd[0]
    return {
        "params": params,
        "time": float(equilibrium.time[time_index]),
        "r": eq_ggd.r[0].values.reshape(nr, nz),
        "z": eq_ggd.z[0].values.reshape(nr, nz),
        "psi": eq_ggd.psi[0].values.reshape(nr, nz),
        "br": eq_ggd.b_field_r[0].values.reshape(nr, nz),
        "bz": eq_ggd.b_field_z[0].values.reshape(nr, nz),
        "bphi": eq_ggd.b_field_phi[0].values.reshape(nr, nz),
    }


def plasma_slice(plasma, time_index=0):
    params = json.loads(str(plasma.code.parameters))
    nr, nz = params["grid_shape"]
    plasma_ggd = plasma.ggd[time_index]
    ion = plasma_ggd.ion[0]
    neutral = plasma_ggd.neutral[0]
    return {
        "params": params,
        "time": float(plasma.time[time_index]),
        "ne": plasma_ggd.electrons.density[0].values.reshape(nr, nz),
        "te": plasma_ggd.electrons.temperature[0].values.reshape(nr, nz),
        "ni": ion.density[0].values.reshape(nr, nz),
        "ti": ion.temperature[0].values.reshape(nr, nz),
        "u_par": ion.velocity[0].parallel.reshape(nr, nz),
        "nn": neutral.density[0].values.reshape(nr, nz),
    }


def plot_field(r, z, field, title, label):
    fig, ax = plt.subplots(figsize=(6, 8))
    im = ax.pcolormesh(r, z, field, shading="auto")
    fig.colorbar(im, ax=ax, label=label)
    ax.set_xlabel("R [m]")
    ax.set_ylabel("Z [m]")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.show()


## 1. Read one steady-state case

A single exported case lives at IDS occurrence `0` and time index `0`.


In [ ]:
steady_db_path = "build/imas_single_case.nc"
summary, equilibrium, plasma = load_ids(steady_db_path, occurrence=0)

summary_params = json.loads(str(summary.code.parameters))
eq = equilibrium_slice(equilibrium, time_index=0)
pl = plasma_slice(plasma, time_index=0)

print("Description:", summary.description)
print("Workflow:", summary.simulation.workflow)
print("Puff rate:", summary_params.get("puff_rate"))
print("Recycling:", summary_params.get("recycling_coefficient"))


In [ ]:
plot_field(eq["r"], eq["z"], eq["psi"], "Steady case: poloidal flux", "psi")
plot_field(eq["r"], eq["z"], pl["ne"], "Steady case: electron density", "n_e")


## 2. Read one simulation from a bundled scan

For the bundled scan export, the scan point is selected by IDS occurrence.


In [ ]:
scan_db_path = "build/puff_scan.nc"
occurrence = 1  # choose the simulation you want from the bundled scan

summary, equilibrium, plasma = load_ids(scan_db_path, occurrence=occurrence)
summary_params = json.loads(str(summary.code.parameters))
eq = equilibrium_slice(equilibrium, time_index=0)
pl = plasma_slice(plasma, time_index=0)

print("Occurrence:", occurrence)
print("Description:", summary.description)
print("Stored export run metadata:", summary_params.get("export_run"))
print("Puff rate:", summary_params.get("puff_rate"))


In [ ]:
plot_field(eq["r"], eq["z"], pl["te"], f"Bundled scan occurrence {occurrence}: electron temperature", "T_e")
plot_field(eq["r"], eq["z"], pl["nn"], f"Bundled scan occurrence {occurrence}: neutral density", "n_n")


## 3. Read one time index from a full discharge

For the full-discharge export, the IDS occurrence stays fixed and the snapshot is selected by `time_index`.


In [ ]:
discharge_db_path = "build/full_discharge.nc"
occurrence = 0
time_index = 0

summary, equilibrium, plasma = load_ids(discharge_db_path, occurrence=occurrence)
summary_params = json.loads(str(summary.code.parameters))
eq = equilibrium_slice(equilibrium, time_index=time_index)
pl = plasma_slice(plasma, time_index=time_index)

print("Description:", summary.description)
print("Workflow:", summary.simulation.workflow)
print("Number of exported snapshots:", len(equilibrium.time))
print("Selected time [s]:", eq["time"])
print("All exported times [s]:", np.asarray(equilibrium.time))


In [ ]:
plot_field(eq["r"], eq["z"], eq["psi"], f"Full discharge time index {time_index}: poloidal flux", "psi")
plot_field(eq["r"], eq["z"], pl["ti"], f"Full discharge time index {time_index}: ion temperature", "T_i")


## Practical notes

- Outside the original HDG mesh, the exporter writes `NaN`.
- The equilibrium and plasma values currently live on the `nodes` subset of the rectangular GGD mesh.
- The stored arrays are flattened in `(R, Z)` order, so the saved `grid_shape` should always be used when reshaping.
- Convention notes such as the interpretation of `psi` and `j_phi` are stored in `equilibrium.code.parameters`.
- For bundled scans, choose the simulation by IDS occurrence.
- For full discharges, choose the snapshot by time index.
